# **Day 65: Unpacking, the Walrus Operator and Advanced match-case**

Today we learn some small features that make code shorter and cleaner.

## **Star Unpacking**

In [1]:
first, *rest = ["Howrah", "Liluah", "Belur", "Bally"]
*others, last = [1, 2, 3, 4]
print(first, rest)
print(others, last)

Howrah ['Liluah', 'Belur', 'Bally']
[1, 2, 3] 4


In [2]:
# Joining lists and dictionaries
north = ["Siliguri", "Darjeeling"]
south = ["Kolkata", "Digha"]
print([*north, *south, "Purulia"])

defaults = {"city": "Kolkata", "lang": "bn"}
user = {"lang": "en", "dark_mode": True}
print({**defaults, **user})

['Siliguri', 'Darjeeling', 'Kolkata', 'Digha', 'Purulia']
{'city': 'Kolkata', 'lang': 'en', 'dark_mode': True}


In [3]:
# Unpacking nested data
record = ("Ananya", (22.57, 88.36), ["Python", "QGIS"])
name, (lat, lon), [skill1, skill2] = record
print(name, lat, skill2)

Ananya 22.57 QGIS


## **The Walrus Operator :=**
The **walrus operator** `:=` **saves a value into a variable** and **uses it** at the same time. (It is called walrus because `:=` looks like the eyes and tusks of a walrus.)

In [4]:
temps = [34, 36, 29, 41, 38, 27]

# Without walrus
n = len(temps)
if n > 5:
    print(f"Too many readings ({n})")

# With walrus: save and check in one line
if (n := len(temps)) > 5:
    print(f"Too many readings ({n})")

Too many readings (6)
Too many readings (6)


In [5]:
# Very handy in while loops
import io
f = io.StringIO("Howrah\nSealdah\nShalimar\n")
while (line := f.readline()):
    print(line.strip())

Howrah
Sealdah
Shalimar


In [6]:
# Avoid calculating something twice in a comprehension
import math
nums = [4, 10, 16, 20, 25, 30, 36]
print([(x, root) for x in nums if (root := math.isqrt(x)) ** 2 == x])     # Perfect squares

[(4, 2), (16, 4), (25, 5), (36, 6)]


## **match-case in Depth**
On Day 9, we used `match` with simple values. It can also look **inside** lists, dictionaries and objects. This is called **pattern matching**.

### **Matching Lists (Commands)**

In [7]:
def handle(command):
    match command.split():
        case ["go", direction]:
            return f"Going {direction}"
        case ["buy", item, qty] if qty.isdigit():
            return f"Buying {qty} x {item}"
        case ["buy", item]:
            return f"Buying 1 x {item}"
        case ["quit" | "exit"]:
            return "Bye"
        case []:
            return "Please type something"
        case _:
            return f"Unknown command {command!r}"

for c in ["go north", "buy rosogolla 10", "buy jhalmuri", "exit", "", "dance"]:
    print(f"{c!r:<20} -> {handle(c)}")

'go north'           -> Going north
'buy rosogolla 10'   -> Buying 10 x rosogolla
'buy jhalmuri'       -> Buying 1 x jhalmuri
'exit'               -> Bye
''                   -> Please type something
'dance'              -> Unknown command 'dance'


### **Example from GIS: Matching GeoJSON Geometries**
GeoJSON geometries are dictionaries. With `match`, we can handle each geometry type in a very clean way.

In [8]:
def describe_geometry(geom):
    match geom:
        case {"type": "Point", "coordinates": [lon, lat]}:
            return f"Point at lat {lat}, lon {lon}"
        case {"type": "LineString", "coordinates": [first, *_, last]}:
            return f"Line from {first} to {last}, {len(geom['coordinates'])} points"
        case {"type": "Polygon", "coordinates": [outer, *holes]}:
            return f"Polygon with {len(outer) - 1} corners and {len(holes)} hole(s)"
        case {"type": other}:
            return f"Geometry type {other} is not handled"
        case _:
            return "Not a geometry"

geometries = [
    {"type": "Point", "coordinates": [88.36, 22.57]},
    {"type": "LineString", "coordinates": [[88.36, 22.57], [88.10, 22.35], [87.51, 21.63]]},
    {"type": "Polygon", "coordinates": [[[0, 0], [4, 0], [4, 3], [0, 3], [0, 0]], [[1, 1], [2, 1], [2, 2], [1, 1]]]},
    {"type": "MultiPoint", "coordinates": [[1, 2], [3, 4]]},
    {"name": "no type here"},
]
for g in geometries:
    print(describe_geometry(g))

Point at lat 22.57, lon 88.36
Line from [88.36, 22.57] to [87.51, 21.63], 3 points
Polygon with 4 corners and 1 hole(s)
Geometry type MultiPoint is not handled
Not a geometry


Let us understand the code above:
- `{"type": "Point", "coordinates": [lon, lat]}` matches only if the type is "Point", and it **pulls out** the two numbers into `lon` and `lat` at the same time.
- `[first, *_, last]` takes the first and the last point of a line.
- `[outer, *holes]`: A polygon's first ring is its outer boundary, and the rest are holes (like a lake inside a park).

### **Matching Objects**

In [9]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

def where(p):
    match p:
        case Point(x=0, y=0):
            return "At the origin"
        case Point(x=0):
            return "On the Y axis"
        case Point(y=0):
            return "On the X axis"
        case Point(x=x, y=y) if x == y:
            return "On the diagonal"
        case _:
            return "Somewhere else"

for p in [Point(0, 0), Point(0, 5), Point(3, 0), Point(2, 2), Point(1, 4)]:
    print(p, "->", where(p))

Point(x=0, y=0) -> At the origin
Point(x=0, y=5) -> On the Y axis
Point(x=3, y=0) -> On the X axis
Point(x=2, y=2) -> On the diagonal
Point(x=1, y=4) -> Somewhere else


## **Practice Problem: A Small Calculator**

In [10]:
def calc(expression):
    match expression.split():
        case [a, "+", b]:
            return float(a) + float(b)
        case [a, "-", b]:
            return float(a) - float(b)
        case [a, "*" | "x", b]:
            return float(a) * float(b)
        case [a, "/", b] if float(b) != 0:
            return float(a) / float(b)
        case [_, "/", _]:
            return "Cannot divide by zero"
        case _:
            return "Bad expression"

for e in ["3 + 4", "10 x 2", "9 / 0", "7 / 2", "hello"]:
    print(e, "=", calc(e))

3 + 4 = 7.0
10 x 2 = 20.0
9 / 0 = Cannot divide by zero
7 / 2 = 3.5
hello = Bad expression


In [11]:
# Quick question
# Guess the output of this code cell
# a, *b, c = [1, 2, 3, 4, 5]
# print(a, b, c)